1. 문서의 내용을 읽는다
2. 문서를 쪼갠다.
    - 토큰수 초과로 답변을 생성하지 못할 수 있고
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래걸림
3. 임베딩 -> 벡터 데이터베이스에 저장
4. 질문이 있을 대, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

loader = Docx2txtLoader("./tax_with_markdown.docx")
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'tax-markdown-index'
pinecone_api_key = os.environ.get("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

# database = PineconeVectorStore.from_documents(document_list, embedding=embedding, index_name=index_name)
database = PineconeVectorStore.from_existing_index(embedding=embedding, index_name=index_name)

In [ ]:
query = '연봉 5000만원인 직장인의 소득세는 얼마인가요?'

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano")

In [ ]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("rlm/rag-prompt")

In [ ]:
prompt.input_variables, prompt.messages[0].prompt.template

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = database.as_retriever()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke(query)

In [ ]:
answer

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["사람을 나타내는 표현 -> 거주자"]

dictionary_prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단되면, 사용자의 질문을 변경하지 않아도 됩니다.
    변경이 필요 없는 경우에는 질문만 반환해주세요.
    사전: {dictionary}
                                          
    질문: {{question}}
""")

dictionary_chain = dictionary_prompt | llm | StrOutputParser()

In [ ]:
new_question = dictionary_chain.invoke({"question": query})

In [ ]:
new_question

In [ ]:
tax_chain = dictionary_chain | rag_chain

In [ ]:
ai_response = tax_chain.invoke({"question": query})

In [ ]:
ai_response